# Notebook 01 · Texto con MiniLM + MongoDB Atlas Vector Search

Este notebook implementa la primera parte del proyecto en **MongoDB Atlas**:

- ingesta de documentos de texto
- chunking con varias estrategias
- embeddings con **MiniLM**
- carga en MongoDB Atlas
- creación de índices para búsqueda vectorial y filtros
- pruebas de recuperación semántica

> **Objetivo:** dejar lista la colección para que el notebook 02 pueda hacer retrieval top-k, construir contexto e integrar un LLM.

## 1. Instalación de dependencias

In [1]:
!pip -q install pymongo[srv] sentence-transformers nltk pandas numpy scikit-learn openai langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 15.1 MB/s eta 0:00:00


## 2. Imports y preparación

In [2]:
import os
import re
import json
import math
import time
import getpass
import warnings
from typing import List, Dict, Optional, Tuple

# Procesamiento de Datos y Álgebra Lineal
import numpy as np
import pandas as pd

# Conectividad y Operaciones NoSQL (MongoDB Atlas)
from pymongo import MongoClient, UpdateOne
from pymongo.errors import PyMongoError
from pymongo.operations import SearchIndexModel

# Modelos de Machine Learning e Inteligencia Artificial
from sentence_transformers import SentenceTransformer
import openai
from openai import OpenAI

# Procesamiento de Lenguaje Natural (NLP) y Chunking
import nltk
from nltk.tokenize import sent_tokenize
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configuraciones Globales de Entorno
warnings.filterwarnings("ignore")

# Descarga automatizada y silenciosa de recursos lingüísticos necesarios
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print("[OK] ¡Todas las dependencias y librerías se han cargado e inicializado correctamente!")

[OK] ¡Todas las dependencias y librerías se han cargado e inicializado correctamente!


## 3. Conexión a MongoDB Atlas

In [16]:
# Credenciales desde secretos de Colab
from google.colab import userdata
MONGODB_URI = userdata.get('MONGO_URI')

DB_NAME = "cookflow"
COLLECTION_NAME = "chunks_embeddings"
VECTOR_INDEX_NAME = "vector_index_texto"

client = MongoClient(MONGODB_URI)
db = client[DB_NAME]
collection = db[COLLECTION_NAME]

print("=== CONEXION EXITOSA A MONGO DB ATLAS ===")
print("Base activa para CookFlow:", DB_NAME)
print("Coleccion de destino activa:", COLLECTION_NAME)
print("Total documentos almacenados en chunks_embeddings actualmente:", collection.count_documents({}))

=== CONEXION EXITOSA A MONGO DB ATLAS ===
Base activa para CookFlow: cookflow
Coleccion de destino activa: chunks_embeddings
Total documentos almacenados en chunks_embeddings actualmente: 242


## 4. Modelo de embeddings

Usaremos `all-MiniLM-L6-v2`, que genera embeddings de 384 dimensiones.

In [4]:
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBEDDING_DIMENSIONS = 384

embedding_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print("Modelo de Embeddings Textuales Cargado:", EMBEDDING_MODEL_NAME)
print("Dimensiones configuradas para el espacio vectorial:", EMBEDDING_DIMENSIONS)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo de Embeddings Textuales Cargado: sentence-transformers/all-MiniLM-L6-v2
Dimensiones configuradas para el espacio vectorial: 384


## 5. Corpus

In [5]:
# Diccionarios de lookup
ingredientes_db = {
    str(i["_id"]): i["nombre"]
    for i in db.ingredientes.find({}, { "_id": 1, "nombre": 1 })
}
usuarios_db = {
    str(u["_id"]): u["nombre"]
    for u in db.perfiles_usuarios.find({}, { "_id": 1, "nombre": 1 })
}

corpus = []

recetas_db = list(db.recetas.find({}, {
    "_id": 1, "titulo": 1, "preparacion": 1,
    "cantidades": 1, "nutricion": 1, "dificultad": 1,
    "tiempo": 1, "idioma": 1, "tags": 1
}))

for r in recetas_db:
    if isinstance(r.get("preparacion"), list):
        texto_prep = " ".join([p["texto"] for p in r["preparacion"] if "texto" in p])
    else:
        texto_prep = str(r.get("preparacion", ""))

    nut = r.get("nutricion", {})
    info_nut = (
        f"Calorias: {nut.get('calorias', 0)} kcal, "
        f"Proteinas: {nut.get('proteinas', 0)}g, "
        f"Grasas: {nut.get('grasas', 0)}g, "
        f"Carbohidratos: {nut.get('carbohidratos', 0)}g por porcion."
    )

    ingredientes_texto = [
        f"{c.get('cantidad', '')} {c.get('unidad', '')} {ingredientes_db.get(str(c.get('ingrediente_id', '')), 'ingrediente desconocido')}"
        for c in r.get("cantidades", [])
    ]

    corpus.append({
        "_id": str(r["_id"]),
        "titulo": r.get("titulo", ""),
        "tipo_documento": "receta",
        "ingredientes": ingredientes_texto,
        "preparacion": texto_prep,
        "info_nutricional": info_nut,
        "dificultad": r.get("dificultad", ""),
        "tiempo": r.get("tiempo", 0),
        "idioma": r.get("idioma", "es"),
        "tags": r.get("tags", [])
    })

historiales_db = list(db.historial_cocciones.find(
    { "notas": { "$exists": True, "$ne": "" } },
    { "_id": 1, "notas": 1, "receta_id": 1, "usuario_id": 1 }
))

for h in historiales_db:
    corpus.append({
        "_id": f"nota_{str(h['_id'])}",
        "tipo_documento": "nota_cocinero",
        "id_receta_referencia": str(h.get("receta_id", "")),
        "autor": usuarios_db.get(str(h.get("usuario_id", "")), "Chef CookFlow"),
        "texto_nota": h["notas"]
    })

recetas = [doc for doc in corpus if doc["tipo_documento"] == "receta"]
notas   = [doc for doc in corpus if doc["tipo_documento"] == "nota_cocinero"]

df_recetas = pd.DataFrame(recetas)
df_notas   = pd.DataFrame(notas)

print(f"Corpus cargado desde MongoDB: {len(corpus)} documentos")
print(f"  - Recetas: {len(recetas)}")
print(f"  - Notas de cocinero: {len(notas)}")
print("\n=== RECETAS ===")
display(df_recetas[["_id", "titulo", "dificultad", "tiempo", "idioma"]].head(10))
print("\n=== NOTAS ===")
display(df_notas[["_id", "id_receta_referencia", "autor"]].head(10))

Corpus cargado desde MongoDB: 85 documentos
  - Recetas: 50
  - Notas de cocinero: 35

=== RECETAS ===


,_id,titulo,dificultad,tiempo,idioma
0,664a1b2c3d4e5f6a7b8c9f01,Ajiaco Bogotano,media,90,es
1,664a1b2c3d4e5f6a7b8c9f02,Bandeja Paisa Tradicional,media,120,es
2,664a1b2c3d4e5f6a7b8c9f03,Pollo al Curry Express,facil,30,es
3,664a1b2c3d4e5f6a7b8c9f04,Sancocho de Gallina Costeno,media,110,es
4,664a1b2c3d4e5f6a7b8c9f05,Ceviche de Camaron,facil,25,es
5,664a1b2c3d4e5f6a7b8c9f06,Pasta Carbonara Clasica,media,25,es
6,664a1b2c3d4e5f6a7b8c9f07,Gazpacho Andaluz,facil,20,es
7,664a1b2c3d4e5f6a7b8c9f08,Risotto de Champinones,dificil,40,es
8,664a1b2c3d4e5f6a7b8c9f09,Caldo de Costilla Bogotano,facil,130,es
9,664a1b2c3d4e5f6a7b8c9f10,Tamales Tolimenses,dificil,180,es



=== NOTAS ===


,_id,id_receta_referencia,autor
0,nota_664a1b2c3d4e5f6a7b8c9c01,664a1b2c3d4e5f6a7b8c9f01,Carlos Mendez
1,nota_664a1b2c3d4e5f6a7b8c9c02,664a1b2c3d4e5f6a7b8c9f03,Sofia Ramirez
2,nota_664a1b2c3d4e5f6a7b8c9c03,664a1b2c3d4e5f6a7b8c9f02,Laura Gomez
3,nota_664a1b2c3d4e5f6a7b8c9c04,664a1b2c3d4e5f6a7b8c9f06,Miguel Torres
4,nota_664a1b2c3d4e5f6a7b8c9c05,664a1b2c3d4e5f6a7b8c9f07,Ana Jimenez
5,nota_664a1b2c3d4e5f6a7b8c9c06,664a1b2c3d4e5f6a7b8c9f08,Diego Vargas
6,nota_664a1b2c3d4e5f6a7b8c9c07,664a1b2c3d4e5f6a7b8c9f15,Valentina Cruz
7,nota_664a1b2c3d4e5f6a7b8c9c08,664a1b2c3d4e5f6a7b8c9f09,Andres Perez
8,nota_664a1b2c3d4e5f6a7b8c9c09,664a1b2c3d4e5f6a7b8c9f05,Camila Ospina
9,nota_664a1b2c3d4e5f6a7b8c9c10,664a1b2c3d4e5f6a7b8c9f17,Felipe Mora


## 6. Definición de Estrategias

In [6]:
# =====================================================================
# ESTRATEGIAS DE CHUNKING - CookFlow RAG
# =====================================================================

# --- ESTRATEGIA A: Fixed-size chunking ---
def fixed_size_chunking(text: str, chunk_size: int = 256, overlap: int = 32) -> List[str]:
    """
    Divide el texto en chunks de tamaño fijo por palabras, con solapamiento.
    """
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks


# --- ESTRATEGIA B: Sentence-aware chunking ---
def sentence_aware_chunking(text: str, max_sentences: int = 5, overlap_sentences: int = 1) -> List[str]:
    """
    Divide el texto respetando límites de oraciones, con solapamiento por oraciones.
    """
    sentences = sent_tokenize(text, language="spanish")
    chunks = []
    start = 0
    while start < len(sentences):
        end = start + max_sentences
        chunk = " ".join(sentences[start:end])
        chunks.append(chunk)
        start += max_sentences - overlap_sentences
    return chunks


# --- ESTRATEGIA C: Semantic chunking ---
def semantic_chunking(
    text: str,
    threshold: float = 0.78,
    min_sentences: int = 1,
    max_sentences: int = 4
) -> List[str]:
    """
    Agrupa oraciones semánticamente similares usando coseno sobre embeddings MiniLM.
    """
    sentences = sent_tokenize(text, language="spanish")
    if len(sentences) == 0:
        return []
    if len(sentences) == 1:
        return sentences

    # Generamos embeddings para cada oración
    embeddings = embedding_model.encode(sentences, normalize_embeddings=True)

    chunks = []
    current_group = [sentences[0]]

    for i in range(1, len(sentences)):
        # Similitud coseno entre la oración actual y la anterior
        sim = float(np.dot(embeddings[i], embeddings[i - 1]))

        # Agrupamos si son similares y no excedemos el máximo
        if sim >= threshold and len(current_group) < max_sentences:
            current_group.append(sentences[i])
        else:
            # Guardamos el grupo actual y comenzamos uno nuevo
            if len(current_group) >= min_sentences:
                chunks.append(" ".join(current_group))
            current_group = [sentences[i]]

    # Último grupo pendiente
    if len(current_group) >= min_sentences:
        chunks.append(" ".join(current_group))

    return chunks if chunks else [text]


print("[OK] Las tres estrategias de chunking han sido definidas correctamente.")

[OK] Las tres estrategias de chunking han sido definidas correctamente.


## 7. Comparación rápida de chunking sobre un documento

In [7]:
# --- PRUEBA VISUAL DE LAS TRES ESTRATEGIAS ---
receta_ejemplo = corpus[0]
nota_ejemplo = notas[0]

print(f"=== ESTRATEGIA A (Fixed-size) — INGREDIENTES: {receta_ejemplo['titulo']} ===")
texto_ing = f"Ingredientes de {receta_ejemplo['titulo']}: " + ", ".join(receta_ejemplo["ingredientes"])
for idx, ch in enumerate(fixed_size_chunking(texto_ing, chunk_size=256, overlap=32), start=1):
    print(f"[{idx}] {ch}")

print(f"\n=== ESTRATEGIA B (Sentence-aware) — PREPARACIÓN: {receta_ejemplo['titulo']} ===")
texto_prep = f"Preparación de {receta_ejemplo['titulo']}: {receta_ejemplo['preparacion']}"
for idx, ch in enumerate(sentence_aware_chunking(texto_prep, max_sentences=5, overlap_sentences=1), start=1):
    print(f"[{idx}] {ch}")

print(f"\n=== ESTRATEGIA C (Semantic) — NOTA: {nota_ejemplo['autor']} ===")
texto_nota = f"Nota de {nota_ejemplo['autor']}: {nota_ejemplo['texto_nota']}"
for idx, ch in enumerate(semantic_chunking(texto_nota, threshold=0.78, min_sentences=1, max_sentences=4), start=1):
    print(f"[{idx}] {ch}")

=== ESTRATEGIA A (Fixed-size) — INGREDIENTES: Ajiaco Bogotano ===
[1] Ingredientes de Ajiaco Bogotano: 500 gr Pechuga de pollo, 300 gr Papa criolla, 200 gr Papa pastusa, 200 gr Papa sabanera, 2 unidad Mazorca de maiz, 10 gr Guascas

=== ESTRATEGIA B (Sentence-aware) — PREPARACIÓN: Ajiaco Bogotano ===
[1] Preparación de Ajiaco Bogotano: Poner el pollo a hervir con agua, cebolla y sal durante 20 minutos Retirar el pollo y desmenuzarlo en trozos medianos En el mismo caldo agregar las tres clases de papa Cocinar a fuego medio 30 minutos hasta que la papa criolla se deshaga Agregar el maiz y las guascas y cocinar 10 minutos mas Servir con crema de leche y alcaparras al lado

=== ESTRATEGIA C (Semantic) — NOTA: Carlos Mendez ===
[1] Nota de Carlos Mendez: Le agregue mas guascas de lo normal y quedo espectacular.
[2] La proxima vez voy a probar con menos sal.


## 8. Construcción del dataset de chunks

Transformamos cada documento en múltiples chunks y agregamos metadatos:
- `codigo_doc`
- `titulo`
- `idioma`
- `fuente`
- `chunk_id`
- `estrategia_chunking`

In [8]:
def build_chunk_records_cookflow(documents: List[Dict]) -> pd.DataFrame:
    rows = []

    for doc in documents:
        doc_id = doc["_id"]
        tipo = doc["tipo_documento"]

        if tipo == "receta":
            titulo = doc["titulo"]

            # 1. Procesar Ingredientes -> Estrategia Fixed
            texto_ingredientes = f"Ingredientes de {titulo}: " + ", ".join(doc["ingredientes"])
            chunks_ing = fixed_size_chunking(texto_ingredientes, chunk_size=256, overlap=32)
            for idx, chunk_text in enumerate(chunks_ing):
                rows.append({
                    "id_documento_fuente": doc_id,
                    "id_receta_referencia": doc_id, # Al ser la receta, se referencia a sí misma
                    "tipo_fuente": tipo,
                    "titulo_receta": titulo,
                    "chunk_id": f"{doc_id}_ing_{idx}",
                    "estrategia_chunking": "fixed-size",
                    "texto_chunk": chunk_text,
                    "n_chars": len(chunk_text)
                })

            # 2. Procesar Información Nutricional -> Estrategia Fixed
            texto_nutricion = f"Información Nutricional de {titulo}: {doc['info_nutricional']}"
            chunks_nut = fixed_size_chunking(texto_nutricion, chunk_size=256, overlap=32)
            for idx, chunk_text in enumerate(chunks_nut):
                rows.append({
                    "id_documento_fuente": doc_id,
                    "id_receta_referencia": doc_id,
                    "tipo_fuente": tipo,
                    "titulo_receta": titulo,
                    "chunk_id": f"{doc_id}_nut_{idx}",
                    "estrategia_chunking": "fixed-size",
                    "texto_chunk": chunk_text,
                    "n_chars": len(chunk_text)
                })

            # 3. Procesar Preparación -> Estrategia Sentence
            texto_preparacion = f"Preparación de {titulo}: {doc['preparacion']}"
            chunks_prep = sentence_aware_chunking(texto_preparacion, max_sentences=5, overlap_sentences=1)
            for idx, chunk_text in enumerate(chunks_prep):
                rows.append({
                    "id_documento_fuente": doc_id,
                    "id_receta_referencia": doc_id,
                    "tipo_fuente": tipo,
                    "titulo_receta": titulo,
                    "chunk_id": f"{doc_id}_prep_{idx}",
                    "estrategia_chunking": "sentence-aware",
                    "texto_chunk": chunk_text,
                    "n_chars": len(chunk_text)
                })

        elif tipo == "nota_cocinero":
            # 4. Procesar Notas de usuarios -> Estrategia Semantic
            texto_nota = f"Nota de {doc['autor']} para la receta {doc['id_receta_referencia']}: {doc['texto_nota']}"
            chunks_nota = semantic_chunking(texto_nota, threshold=0.78, min_sentences=1, max_sentences=4)
            for idx, chunk_text in enumerate(chunks_nota):
                rows.append({
                    "id_documento_fuente": doc_id,
                    "id_receta_referencia": doc["id_receta_referencia"], # Referencing NoSQL directo
                    "tipo_fuente": tipo,
                    "titulo_receta": f"Nota de {doc['autor']}",
                    "chunk_id": f"{doc_id}_nota_{idx}",
                    "estrategia_chunking": "semantic",
                    "texto_chunk": chunk_text,
                    "n_chars": len(chunk_text)
                })

    return pd.DataFrame(rows)

# Ejecutamos la construcción del dataset fragmentado
df_chunks = build_chunk_records_cookflow(corpus)
df_chunks.head(10)


# === ANÁLISIS ESTÁTICO DE CHUNKS GENERADOS ===
print("=== DISTRIBUCIÓN DE CHUNKS POR ESTRATEGIA ===")
resumen = (
    df_chunks.groupby("estrategia_chunking")
    .agg(
        cantidad=("chunk_id", "count"),
        longitud_promedio=("n_chars", "mean"),
        longitud_min=("n_chars", "min"),
        longitud_max=("n_chars", "max")
    )
    .round(1)
    .reset_index()
)
display(resumen)

=== DISTRIBUCIÓN DE CHUNKS POR ESTRATEGIA ===


,estrategia_chunking,cantidad,longitud_promedio,longitud_min,longitud_max
0,fixed-size,100,119.5,46,188
1,semantic,92,83.8,21,172
2,sentence-aware,50,345.0,265,415


## 9. Generación de embeddings para cada chunk

In [9]:
def embed_texts(texts: List[str], batch_size: int = 32) -> np.ndarray:
    return embedding_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

# Vectorizamos la columna "texto_chunk" usando MiniLM (generará arrays de 384 dimensiones)
chunk_embeddings = embed_texts(df_chunks["texto_chunk"].tolist())
print(f"\nMatriz de embeddings generada con éxito.")
print(f"Dimensiones de la matriz (N Chunks, Dimensiones del Vector): {chunk_embeddings.shape}")

Batches:   0%|          | 0/8 [00:00<?, ?it/s]


Matriz de embeddings generada con éxito.
Dimensiones de la matriz (N Chunks, Dimensiones del Vector): (242, 384)


## 10. Preparación de documentos para MongoDB

Guardamos un documento por chunk.  
Cada documento contiene:

- metadatos del documento fuente
- texto del chunk
- embedding

In [10]:
# Diccionario para lookup rapido de metadatos por id de documento
corpus_meta = {
    doc["_id"]: {
        "dificultad": doc.get("dificultad", ""),
        "calorias": doc.get("nutricion", {}).get("calorias", 0) if isinstance(doc.get("nutricion"), dict) else 0,
        "tags": doc.get("tags", []),
        "idioma": doc.get("idioma", "es"),
        "tiempo": doc.get("tiempo", 0)
    }
    for doc in corpus if doc["tipo_documento"] == "receta"
}

atlas_docs = []
for row, emb in zip(df_chunks.to_dict(orient="records"), chunk_embeddings):
    row["embedding"]      = [float(x) for x in emb]
    row["n_tokens_aprox"] = max(1, math.ceil(len(row["texto_chunk"].split()) * 1.2))
    row["chunk_texto"]    = row["texto_chunk"]
    row["doc_id"]         = row["id_documento_fuente"]
    row["chunk_index"]    = 0
    row["modelo"]         = "all-MiniLM-L6-v2"

    # Metadatos para filtros hibridos
    meta = corpus_meta.get(row["id_documento_fuente"], {})
    if meta:
        row["meta"] = {
            "dificultad": meta["dificultad"],
            "calorias":   meta["calorias"],
            "tags":       meta["tags"],
            "idioma":     meta["idioma"],
            "tiempo":     meta["tiempo"]
        }

    atlas_docs.append(row)

atlas_docs[0]

{'id_documento_fuente': '664a1b2c3d4e5f6a7b8c9f01',
 'id_receta_referencia': '664a1b2c3d4e5f6a7b8c9f01',
 'tipo_fuente': 'receta',
 'titulo_receta': 'Ajiaco Bogotano',
 'chunk_id': '664a1b2c3d4e5f6a7b8c9f01_ing_0',
 'estrategia_chunking': 'fixed-size',
 'texto_chunk': 'Ingredientes de Ajiaco Bogotano: 500 gr Pechuga de pollo, 300 gr Papa criolla, 200 gr Papa pastusa, 200 gr Papa sabanera, 2 unidad Mazorca de maiz, 10 gr Guascas',
 'n_chars': 161,
 'embedding': [0.027889801189303398,
  -0.004414542578160763,
  -0.024837873876094818,
  -0.00964469462633133,
  0.011393982917070389,
  0.025638651102781296,
  0.0034024014603346586,
  0.015451972372829914,
  -0.030430082231760025,
  -0.04471677914261818,
  0.05121634528040886,
  -0.07637960463762283,
  -0.09658893197774887,
  -0.07717965543270111,
  -0.02528812363743782,
  -0.034034792333841324,
  0.08065911382436752,
  0.06588270515203476,
  0.002320445142686367,
  -0.04740309715270996,
  0.05349719151854515,
  -0.10866918414831161,
  -0.00

## 11. Carga en MongoDB Atlas

Usamos `upsert` por combinación de:
- `codigo_doc`
- `estrategia_chunking`
- `chunk_id`

Así puedes reejecutar el notebook sin duplicar chunks.

In [17]:
def load_chunks_to_mongodb(collection, docs: List[Dict]):
    ops = []
    for doc in docs:
        # Filtro único basado en tu nuevo esquema de CookFlow
        filt = {
            "id_documento_fuente": doc["id_documento_fuente"],
            "chunk_id": doc["chunk_id"]
        }
        ops.append(
            UpdateOne(
                filt,
                {"$set": doc},
                upsert=True  # Si no existe lo crea, si existe lo actualiza
            )
        )

    if not ops:
        return None

    result = collection.bulk_write(ops, ordered=False)
    return result

# Ejecutamos la inserción en la colección de chunks
result = load_chunks_to_mongodb(collection, atlas_docs)
print(result.bulk_api_result if result else "Sin operaciones")
print("Total documentos en colección de chunks:", collection.count_documents({}))

{'writeErrors': [], 'writeConcernErrors': [], 'nInserted': 0, 'nUpserted': 0, 'nMatched': 242, 'nModified': 0, 'nRemoved': 0, 'upserted': []}
Total documentos en colección de chunks: 242


## 12. Funciones de búsqueda semántica

Usaremos `$vectorSearch` para recuperar los chunks más cercanos semánticamente.

In [18]:
def embed_query(question: str) -> List[float]:
    return embedding_model.encode(question, normalize_embeddings=True).tolist()


def search_top_k(
    collection,
    question_vector: List[float],
    top_k: int = 5,
    strategy: Optional[str] = None,
    tipo_fuente: Optional[str] = None,
    num_candidates: int = 100
) -> pd.DataFrame:

    # Configuramos el operador principal de Atlas Vector Search
    vector_stage = {
        "$vectorSearch": {
            "index": VECTOR_INDEX_NAME,
            "path": "embedding",
            "queryVector": question_vector,
            "numCandidates": max(num_candidates, top_k),
            "limit": top_k
        }
    }

    # Construcción dinámica de filtros híbridos requerida por la rúbrica
    filters = []
    if strategy:
        filters.append({"estrategia_chunking": {"$eq": strategy}})
    if tipo_fuente:
        filters.append({"tipo_fuente": {"$eq": tipo_fuente}})

    if filters:
        vector_stage["$vectorSearch"]["filter"] = {"$and": filters} if len(filters) > 1 else filters[0]

    # Pipeline de agregación NoSQL
    pipeline = [
        vector_stage,
        {
            "$project": {
                "_id": 0,
                "id_documento_fuente": 1,
                "id_receta_referencia": 1,
                "tipo_fuente": 1,
                "titulo_receta": 1,
                "chunk_id": 1,
                "estrategia_chunking": 1,
                "texto_chunk": 1,
                "score": {"$meta": "vectorSearchScore"} # Extrae la similitud coseno calculada por Atlas
            }
        }
    ]

    rows = list(collection.aggregate(pipeline))
    return pd.DataFrame(rows)

## 13. Prueba de búsqueda semántica

Cambia la consulta y observa:
- qué chunks recupera
- cómo cambia el score
- qué estrategia funciona mejor

In [19]:
# Hacemos interactiva la sección de consultas del usuario
# "recetas colombianas tradicionales con pollo"
query = input("¿Qué deseas buscar en el asistente de cocina CookFlow?: ").strip()

if query:
    query_vector = embed_query(query)

    df_results = search_top_k(
        collection=collection,
        question_vector=query_vector,
        top_k=5,
        strategy=None,
        tipo_fuente=None
    )

    # Mostrar la tabla de resultados recuperados de Atlas Vector Search
    print(f"\nResultados principales para: '{query}'")

    # Aplicamos estilos para que 'texto_chunk' y todas las celdas se vean completas y alineadas a la izquierda
    df_estilizado = df_results.style.set_properties(**{
        'text-align': 'left',
        'white-space': 'normal',
        'max-width': '600px'  # Ancho límite para que no se estire infinito hacia los lados
    })

    display(df_estilizado)
else:
    print("No ingresaste ninguna consulta.")

¿Qué deseas buscar en el asistente de cocina CookFlow?: sushi

Resultados principales para: 'sushi'


,id_documento_fuente,chunk_id,estrategia_chunking,id_receta_referencia,texto_chunk,tipo_fuente,titulo_receta,score
0,664a1b2c3d4e5f6a7b8c9f18,664a1b2c3d4e5f6a7b8c9f18_ing_0,fixed-size,664a1b2c3d4e5f6a7b8c9f18,"Ingredientes de Sushi Rolls de Salmon: 300 gr Salmon grado sashimi, 4 unidad Alga nori, 200 gr Arroz blanco, 1 unidad Aguacate Hass, 1 unidad Pepino",receta,Sushi Rolls de Salmon,0.767868
1,664a1b2c3d4e5f6a7b8c9f18,664a1b2c3d4e5f6a7b8c9f18_nut_0,fixed-size,664a1b2c3d4e5f6a7b8c9f18,"Información Nutricional de Sushi Rolls de Salmon: Calorias: 340 kcal, Proteinas: 22g, Grasas: 12g, Carbohidratos: 38g por porcion.",receta,Sushi Rolls de Salmon,0.764480
2,664a1b2c3d4e5f6a7b8c9f18,664a1b2c3d4e5f6a7b8c9f18_prep_0,sentence-aware,664a1b2c3d4e5f6a7b8c9f18,"Preparación de Sushi Rolls de Salmon: Preparar el arroz con vinagre de arroz, azucar y sal Dejar enfriar el arroz a temperatura ambiente Extender el arroz sobre el alga nori dejando un borde libre Poner una linea de salmon, aguacate y pepino en el centro Enrollar con la esterilla de bambu firmemente Cortar en 8 piezas con cuchillo humedo",receta,Sushi Rolls de Salmon,0.728360
3,nota_664a1b2c3d4e5f6a7b8c9c16,nota_664a1b2c3d4e5f6a7b8c9c16_nota_0,semantic,664a1b2c3d4e5f6a7b8c9f18,Nota de Diego Vargas para la receta 664a1b2c3d4e5f6a7b8c9f18: El sushi casero requiere practica para el enrollado.,nota_cocinero,Nota de Diego Vargas,0.721060
4,664a1b2c3d4e5f6a7b8c9f43,664a1b2c3d4e5f6a7b8c9f43_ing_0,fixed-size,664a1b2c3d4e5f6a7b8c9f43,Ingredientes de Soup Dumplings Xiao Long Bao: 4 unidad Alga nori,receta,Soup Dumplings Xiao Long Bao,0.661154


## 14. Construcción de contexto base para el siguiente notebook

Este bloque deja lista una función simple para convertir resultados en contexto.

In [20]:
def build_context_from_results(df_results: pd.DataFrame, max_chars: int = 2500) -> str:
    blocks = []
    current_len = 0

    if df_results is None or df_results.empty:
        return ""

    for i, row in df_results.iterrows():
        # Construimos el bloque adaptado a los metadatos gastronómicos de CookFlow
        block = (
            f"[Fragmento {i+1}]\n"
            f"Referencia Receta: {row['titulo_receta']}\n"
            f"Tipo de Información: {row['tipo_fuente']}\n"
            f"Estrategia aplicada: {row['estrategia_chunking']}\n"
            f"Similitud Coseno (Score): {float(row['score']):.4f}\n"
            f"Contenido Extraído:\n{row['texto_chunk']}"
        )

        if current_len + len(block) > max_chars:
            break

        blocks.append(block)
        current_len += len(block) + 2

    return "\n\n".join(blocks)

# Probamos la impresión del contexto consolidado usando el resultado anterior
contexto_generado = build_context_from_results(df_results, max_chars=1200)
print(contexto_generado)

[Fragmento 1]
Referencia Receta: Sushi Rolls de Salmon
Tipo de Información: receta
Estrategia aplicada: fixed-size
Similitud Coseno (Score): 0.7679
Contenido Extraído:
Ingredientes de Sushi Rolls de Salmon: 300 gr Salmon grado sashimi, 4 unidad Alga nori, 200 gr Arroz blanco, 1 unidad Aguacate Hass, 1 unidad Pepino

[Fragmento 2]
Referencia Receta: Sushi Rolls de Salmon
Tipo de Información: receta
Estrategia aplicada: fixed-size
Similitud Coseno (Score): 0.7645
Contenido Extraído:
Información Nutricional de Sushi Rolls de Salmon: Calorias: 340 kcal, Proteinas: 22g, Grasas: 12g, Carbohidratos: 38g por porcion.

[Fragmento 3]
Referencia Receta: Sushi Rolls de Salmon
Tipo de Información: receta
Estrategia aplicada: sentence-aware
Similitud Coseno (Score): 0.7284
Contenido Extraído:
Preparación de Sushi Rolls de Salmon: Preparar el arroz con vinagre de arroz, azucar y sal Dejar enfriar el arroz a temperatura ambiente Extender el arroz sobre el alga nori dejando un borde libre Poner una lin

## 15. Comparación básica entre estrategias para una misma consulta

In [21]:
# =====================================================================
# EXPERIMENTO COMPARATIVO DE CHUNKING — 10 CONSULTAS FIJAS
# =====================================================================

CONSULTAS_PRUEBA = [
    "Recetas con pollo y menos de 400 calorias",
    "Como se hace una buena salsa de tomate casera?",
    "Opciones sin gluten para el desayuno",
    "Recetas faciles para principiantes con arroz",
    "Que postre puedo hacer en menos de 20 minutos?",
    "Platos vegetarianos ricos en proteinas",
    "Recetas tipicas de Colombia para una cena especial",
    "Algo facil para sorprender a mis invitados",
    "Que puedo cocinar con zanahoria, cebolla y ajo?",
    "Recetas saludables valoradas con mas de 4 estrellas",
]

# --- Parte 1: Tabla comparativa de scores por estrategia ---
filas_comparacion = []

for pregunta in CONSULTAS_PRUEBA:
    qvec = embed_query(pregunta)
    for strategy in ["fixed-size", "sentence-aware", "semantic"]:
        df_tmp = search_top_k(
            collection=collection,
            question_vector=qvec,
            top_k=3,
            strategy=strategy,
            tipo_fuente=None
        )
        filas_comparacion.append({
            "Consulta": pregunta[:60] + "..." if len(pregunta) > 60 else pregunta,
            "Estrategia": strategy,
            "Top1 Score": float(df_tmp.iloc[0]["score"]) if not df_tmp.empty else None,
            "Top1 Fragmento": df_tmp.iloc[0]["texto_chunk"][:100] + "..." if not df_tmp.empty else "Sin resultado",
        })

df_comparacion = pd.DataFrame(filas_comparacion)
print("=== TABLA COMPARATIVA — 10 CONSULTAS × 3 ESTRATEGIAS ===")
display(df_comparacion)

# --- Parte 2: Análisis de cantidad y longitud promedio por estrategia ---
print("\n=== ANÁLISIS DE CHUNKS EN ATLAS ===")
pipeline_stats = [
    {"$group": {
        "_id": "$estrategia_chunking",
        "cantidad": {"$sum": 1},
        "longitud_promedio": {"$avg": "$n_chars"},
        "longitud_min": {"$min": "$n_chars"},
        "longitud_max": {"$max": "$n_chars"}
    }},
    {"$sort": {"_id": 1}}
]
stats = list(collection.aggregate(pipeline_stats))
df_stats = pd.DataFrame(stats).rename(columns={"_id": "estrategia_chunking"})
df_stats["longitud_promedio"] = df_stats["longitud_promedio"].round(1)
display(df_stats)

# --- Parte 3: Score promedio por estrategia (resumen ejecutivo) ---
print("\n=== SCORE PROMEDIO POR ESTRATEGIA ===")
df_resumen = (
    df_comparacion.dropna(subset=["Top1 Score"])
    .groupby("Estrategia")["Top1 Score"]
    .agg(score_promedio="mean", score_max="max", score_min="min")
    .round(4)
    .reset_index()
)
display(df_resumen)

=== TABLA COMPARATIVA — 10 CONSULTAS × 3 ESTRATEGIAS ===


,Consulta,Estrategia,Top1 Score,Top1 Fragmento
0,Recetas con pollo y menos de 400 calorias,fixed-size,0.741795,Ingredientes de Locro Argentino: 300 gr Mazorc...
1,Recetas con pollo y menos de 400 calorias,sentence-aware,0.763514,Preparación de Cazuela de Mariscos: Sofreir ce...
2,Recetas con pollo y menos de 400 calorias,semantic,0.780154,El caldo queda con un sabor profundo que el po...
3,Como se hace una buena salsa de tomate casera?,fixed-size,0.776942,Ingredientes de Chili con Carne Texano: 400 gr...
4,Como se hace una buena salsa de tomate casera?,sentence-aware,0.805098,Preparación de Crepes Suzette: Preparar la mas...
5,Como se hace una buena salsa de tomate casera?,semantic,0.840763,Con pan artesanal para mojar en la salsa es pe...
6,Opciones sin gluten para el desayuno,fixed-size,0.755470,Información Nutricional de Tacos de Carnitas M...
7,Opciones sin gluten para el desayuno,sentence-aware,0.742312,Preparación de Tacos de Carnitas Mexicanos: Co...
8,Opciones sin gluten para el desayuno,semantic,0.748121,El vino blanco seco es imprescindible para des...
9,Recetas faciles para principiantes con arroz,fixed-size,0.728716,Ingredientes de Arroz con Leche Colombiano: 20...



=== ANÁLISIS DE CHUNKS EN ATLAS ===


,estrategia_chunking,cantidad,longitud_promedio,longitud_min,longitud_max
0,fixed-size,100,119.5,46,188
1,semantic,92,83.8,21,172
2,sentence-aware,50,345.0,265,415



=== SCORE PROMEDIO POR ESTRATEGIA ===


,Estrategia,score_promedio,score_max,score_min
0,fixed-size,0.7477,0.7769,0.7150
1,semantic,0.7944,0.8663,0.7481
2,sentence-aware,0.7724,0.8354,0.7235


## 16. Inspección rápida de la colección

Puedes revisar algunos documentos cargados.

In [22]:
# Hacemos un find básico a MongoDB para verificar la persistencia de los chunks
sample_docs = list(collection.find({}, {
    "_id": 0,
    "id_documento_fuente": 1,
    "titulo_receta": 1,
    "estrategia_chunking": 1,
    "chunk_id": 1,
    "texto_chunk": 1
}).limit(5))

# Visualizamos la muestra en formato tabular
pd.DataFrame(sample_docs)

,chunk_id,id_documento_fuente,estrategia_chunking,texto_chunk,titulo_receta
0,664a1b2c3d4e5f6a7b8c9f01_ing_0,664a1b2c3d4e5f6a7b8c9f01,fixed-size,Ingredientes de Ajiaco Bogotano: 500 gr Pechug...,Ajiaco Bogotano
1,664a1b2c3d4e5f6a7b8c9f01_nut_0,664a1b2c3d4e5f6a7b8c9f01,fixed-size,Información Nutricional de Ajiaco Bogotano: Ca...,Ajiaco Bogotano
2,664a1b2c3d4e5f6a7b8c9f01_prep_0,664a1b2c3d4e5f6a7b8c9f01,sentence-aware,Preparación de Ajiaco Bogotano: Poner el pollo...,Ajiaco Bogotano
3,664a1b2c3d4e5f6a7b8c9f02_ing_0,664a1b2c3d4e5f6a7b8c9f02,fixed-size,Ingredientes de Bandeja Paisa Tradicional: 200...,Bandeja Paisa Tradicional
4,664a1b2c3d4e5f6a7b8c9f02_nut_0,664a1b2c3d4e5f6a7b8c9f02,fixed-size,Información Nutricional de Bandeja Paisa Tradi...,Bandeja Paisa Tradicional
